# COVID-19：Program Identity → Enrichment → Usage → Direction → Integration

本 Notebook 使用新版 Layer0–3 接口；同一张基因表从身份标注沿用到最终组内驱动标注。
富集和 usage 独立，方向标注随后运行。此文件是最终维护入口；历史实现可从 Git 历史查看。

运行前安装项目依赖，选择正确 Python 内核并把工作目录设为仓库根目录。
真实数据没有随仓库提供，下面的主流程需要 cNMF/H5AD/注释输入；wPSI 缺失时明确跳过方向分析。
完整输出写入 `analysis_results/`，不覆盖旧的 `outputs/`。

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display
from depa_downstream import (
    load_cnmf_outputs, build_program_gene_tables, run_functional_enrichment,
    compute_contrast_association, summarize_program_associations,
    compute_population_direction, annotate_within_celltype,
    plot_enrichment_bubble, plot_association_heatmap, plot_program_umap_grid,
    plot_population_direction, plot_within_celltype_concordance,
)
from depa_downstream.pipeline import load_analysis_config, assemble_analysis_results, export_analysis

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'pyproject.toml').is_file():
    raise RuntimeError('请先把 Notebook 工作目录切换到 DEPA_downstream 仓库根目录')

## 配置与数据加载
只加载一次配置；后续单元不再重新读 YAML，也不再手动指定 program→celltype。

In [ ]:
# DATA_ROOT 是包含 data/covid_19_data/ 的目录。
DATA_ROOT = PROJECT_ROOT
CONFIG = load_analysis_config(
    PROJECT_ROOT / 'configs' / 'covid19.yaml', root_dir=DATA_ROOT,
    output_root=PROJECT_ROOT / 'analysis_results' / 'covid19',
    path_overrides={'gene_symbol_map_path': str(PROJECT_ROOT / 'configs' / 'covid19_gene_symbol_map.csv')},
)

OUTPUT_ROOT = Path(CONFIG['output_root'])
data = load_cnmf_outputs(CONFIG)

## Layer 0 · Program Identity
PAC→gene 按最大权重和最大 Z 聚合。`is_core` 使用 ≥ 阈值；低 wPSI 检出率只标注，不删除核心基因。后续各层只读取这些标志。

In [ ]:
gene_tables = build_program_gene_tables(data, CONFIG)
identity_summary = pd.DataFrame({p: {'n_genes': len(t), 'n_core': int(t.is_core.sum()), 'n_specific': int(t.is_specific.sum()), 'n_reliable': int(t.is_wpsi_reliable.fillna(False).sum())} for p,t in gene_tables.items()}).T
display(identity_summary)

## Layer 1 · Functional Enrichment
分别对核心基因和专属 marker 查询；背景为当前 PAC 基因全集的可用 symbol，使用 Speedrichr 自定义背景接口。
失败、结果为空和基因太少分别标记；无显著富集不自动判定 cNMF 失败。此时不使用方向性信息。

In [ ]:
enrichment = run_functional_enrichment(gene_tables, CONFIG, on_error='record')
enrichment_status = pd.DataFrame([dict(program=p, gene_set=k, **{a:v for a,v in t.attrs.items() if a not in ('program','gene_set')}) for p,sets in enrichment.items() for k,t in sets.items()])
display(enrichment_status)
for kind in ('core', 'specific'):
    plot_enrichment_bubble({p: sets[kind] for p,sets in enrichment.items()}, str(OUTPUT_ROOT / 'layer1'), filename=f'layer1_enrichment_{kind}.png')

## Layer 2 · Usage Association
仅使用 cells×program 的 usage 与 metadata。最佳细胞类型按 |AUC−0.5| 选取；同分按类型名排序，低 AUC 表示耗竭关联而非富集。

In [ ]:
assoc_results = compute_contrast_association({'usage': data['usage'], 'meta': data['meta']}, CONFIG)
association_summary = summarize_program_associations(assoc_results, CONFIG, programs=list(gene_tables))
display(association_summary)
plot_association_heatmap(assoc_results, CONFIG, str(OUTPUT_ROOT / 'layer2'))
plot_program_umap_grid(data, CONFIG, str(OUTPUT_ROOT / 'layer2'))

## Layer 3a · 群体方向检验
读取 Layer0 缓存的可靠基因中位数；默认背景是可靠 PAC universe 中剔除当前前景的基因，与旧版 MWU 的不重叠比较一致。
`expected_direction` 明确表示 usage **升高**时预期的 wPSI 增减，不等同于疾病组 B−A 的符号。
方向是群体前景相对背景的描述，`is_significant` 独立记录 BH 显著性。

In [ ]:
population_direction = compute_population_direction(gene_tables, CONFIG)
display(population_direction)
plot_population_direction(population_direction, str(OUTPUT_ROOT / 'layer3'), filename='layer3_population_direction.png')

## Layer 3b · 自动选择细胞类型并写回基因表
逐个 program 使用 Layer2 选出的细胞类型，按疾病两组检验全部核心基因。
`within_celltype_mean_diff` 固定为 B−A；一致性比较会乘以该细胞类型内 usage(B−A) 的符号，避免把 usage 下降的 program 判反。
细胞级 MWU 用于探索性标注，不能替代 donor/sample 级独立重复验证；未检验项与不确定项分开保留。

In [ ]:
annotate_within_celltype(gene_tables, data, CONFIG, association_summary, population_direction)
concordance_summary = pd.DataFrame({p: t.loc[t.is_core, 'concordance'].value_counts() for p,t in gene_tables.items()}).fillna(0).astype(int).T
display(concordance_summary)
for program in gene_tables:
    plot_within_celltype_concordance(gene_tables, program, str(OUTPUT_ROOT / 'layer3'))

## 收尾整合 · Enrichment × Concordance
不重新富集、不重新筛选核心基因。对每个 term 追加 concordant/discordant/inconclusive 的数量、symbol 和 gene ID；未检验和未匹配的命中单独记录。

In [ ]:
results = assemble_analysis_results(gene_tables, enrichment, assoc_results, association_summary, population_direction, CONFIG)
integrated_enrichment = results['integrated_enrichment']
export_analysis(results, OUTPUT_ROOT)
print('分析执行状态:', results['manifest']['status'])
print('结果目录:', OUTPUT_ROOT)
report_terms = pd.concat([t.assign(program=p, gene_set=k) for p,sets in integrated_enrichment.items() for k,t in sets.items()], ignore_index=True)
if not report_terms.empty:
    display(report_terms.sort_values('padj').head(30))

## 既有 P03 样本级补充分析
保留下方样本级方法作为独立的生物学重复检查；核心基因直接引用 Layer0，不重复计算权重和筛选阈值。

## P03：样本级疾病关联与候选 APA 分析
运行前需完成配置、数据加载和 Layer0 的 `gene_tables`；本节不依赖 Layer1–3 的中间变量。
默认将 sample 当作独立生物学重复；若多个 sample 来自同一供者，请将 P03_UNIT 改成供者列。
所有统计为探索性：输入 PAC 已经过特征筛选，尚无外部验证。
主分析使用每细胞归一化 usage（六个 program 总和为 1），并输出原始 usage 敏感性分析。
wPSI 部分是已有单细胞比例的样本均值，不是从原始 PAC 计数重算的 pseudobulk APA；不能替代覆盖度控制的最终 APA 检验。

In [ ]:
from pathlib import Path
from itertools import combinations
import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

assert 'data' in globals() and 'CONFIG' in globals(), '请先运行 CONFIG 和 load_cnmf_outputs 单元格'
P03_UNIT = CONFIG['meta_columns']['sample']  # 若有重复供者，改成 donor 列
P03_CT = CONFIG['meta_columns']['celltype']
P03_GROUP = 'group'
P03_A, P03_B = 'Healthy', 'Severe_COVID'
P03_MIN_CELLS = 20
P03_MIN_UNITS = 3  # 每组至少3个有效独立样本，否则只展示，不推断
P03_TARGETS = ['cDC2', 'ncMono', 'cMono']  # 预先指定，不按本次显著性挑选
P03_MIN_VALID_WPSI = 20  # 非缺失wPSI细胞数，并非原始read覆盖度
P03_MIN_VALID_FRAC = 0.2
P03_SEED = 20260928
p03_root = Path(CONFIG['root_dir']).expanduser().resolve()
p03_out = Path(CONFIG['output_root']).expanduser()
if not p03_out.is_absolute():
    p03_out = p03_root / p03_out
p03_out = p03_out / 'p03_disease'
p03_out.mkdir(parents=True, exist_ok=True)

def p03_bh(values):
    p = np.asarray(values, dtype=float)
    q = np.full(len(p), np.nan)
    ok = np.flatnonzero(np.isfinite(p))
    if len(ok):
        order = ok[np.argsort(p[ok])]
        adjusted = p[order] * len(order) / np.arange(1, len(order)+1)
        q[order] = np.minimum(1, np.minimum.accumulate(adjusted[::-1])[::-1])
    return q

def p03_test(a, b, seed=P03_SEED):
    # 双侧样本标签置换；9个样本时枚举全部126种分组，不置换单个细胞。
    a, b = np.asarray(a, float), np.asarray(b, float)
    a, b = a[np.isfinite(a)], b[np.isfinite(b)]
    result = dict(n_healthy=len(a), n_severe=len(b),
                  mean_healthy=np.mean(a) if len(a) else np.nan,
                  mean_severe=np.mean(b) if len(b) else np.nan,
                  delta=np.mean(b)-np.mean(a) if len(a) and len(b) else np.nan,
                  pvalue=np.nan, ci_low=np.nan, ci_high=np.nan,
                  loo_min=np.nan, loo_max=np.nan, loo_same_sign=np.nan,
                  status='insufficient_samples')
    if min(len(a), len(b)) < P03_MIN_UNITS:
        return result
    rng = np.random.default_rng(seed)
    x = np.r_[a,b]; n = len(x); k = len(b)
    def effect(ix):
        mask = np.zeros(n, bool); mask[list(ix)] = True
        return x[mask].mean() - x[~mask].mean()
    total = math.comb(n,k)
    if total <= 50000:
        null = np.array([effect(ix) for ix in combinations(range(n), k)])
        p = np.mean(np.abs(null) >= abs(result['delta'])-1e-12)
    else:
        null = np.array([effect(rng.choice(n,k,replace=False)) for _ in range(9999)])
        p = (1+np.sum(np.abs(null) >= abs(result['delta'])-1e-12))/(1+len(null))
    boot = (rng.choice(b, (4000,len(b)), replace=True).mean(axis=1)
            - rng.choice(a, (4000,len(a)), replace=True).mean(axis=1))
    loo = np.r_[[b.mean()-np.delete(a,i).mean() for i in range(len(a))],
                [np.delete(b,i).mean()-a.mean() for i in range(len(b))]]
    lo, hi = np.quantile(boot,[.025,.975])
    result.update(pvalue=p,ci_low=lo,ci_high=hi,loo_min=loo.min(),loo_max=loo.max(),
                  loo_same_sign=bool(np.all(np.sign(loo)==np.sign(result['delta']))),status='tested')
    return result

p03_meta = data['meta'].copy()
p03_usage = data['usage'].copy()
assert p03_meta.index.is_unique and p03_usage.index.is_unique, '细胞ID必须唯一'
assert all(c in p03_meta for c in [P03_UNIT,P03_CT,P03_GROUP])
assert 'P03' in p03_usage
p03_common = p03_meta.index.intersection(p03_usage.index)
p03_cells = p03_meta.loc[p03_common,[P03_UNIT,P03_CT,P03_GROUP]].dropna().copy()
p03_cells = p03_cells[p03_cells[P03_GROUP].isin([P03_A,P03_B])]
assert p03_cells.groupby(P03_UNIT,observed=True)[P03_GROUP].nunique().max()==1, '同一分析单位跨疾病组：需要重新定义设计'
p03_values = p03_usage.loc[p03_cells.index].astype(float)
assert np.isfinite(p03_values.to_numpy()).all() and (p03_values.to_numpy()>=0).all()
p03_sums = p03_values.sum(axis=1)
print('剔除零总usage细胞:', int((p03_sums<=0).sum()))
p03_cells = p03_cells.loc[p03_sums>0].copy()
p03_cells['P03_raw'] = p03_values.loc[p03_cells.index,'P03']
p03_cells['P03_fraction'] = p03_values.loc[p03_cells.index,'P03']/p03_sums.loc[p03_cells.index]
p03_keys = [P03_UNIT,P03_GROUP,P03_CT]
p03_summary = p03_cells.groupby(p03_keys,observed=True).agg(
    n_cells=('P03_fraction','size'),P03_fraction=('P03_fraction','mean'),P03_raw=('P03_raw','mean')).reset_index()
p03_summary.to_csv(p03_out/'sample_celltype_usage.csv',index=False)
p03_counts = pd.crosstab(p03_cells[P03_UNIT],p03_cells[P03_CT])
p03_counts.to_csv(p03_out/'sample_celltype_counts.csv')
p03_composition = p03_counts.div(p03_counts.sum(axis=1),axis=0)
p03_composition.to_csv(p03_out/'sample_celltype_proportions.csv')
print('输出目录:',p03_out)
display(p03_cells[[P03_UNIT,P03_GROUP]].drop_duplicates().sort_values([P03_GROUP,P03_UNIT]))
display(p03_counts)

### 1. 样本级 usage 差异、FDR 与敏感性
每个样本等权。FDR 分别在每个 usage 口径和细胞数门槛下、所有被检验细胞类型间校正。
bootstrap 区间是小样本探索性区间；精确置换 p 值较离散，不能保证在当前样本数下通过 FDR。
`loo_same_sign` 表示逐个删除样本后效应方向均保留，不代表每次都显著。

In [ ]:
p03_rows = []
for cutoff in sorted(set([P03_MIN_CELLS,30,50])):
    for metric in ['P03_fraction','P03_raw']:
        for ct, frame in p03_summary.groupby(P03_CT,observed=True):
            valid = frame[frame.n_cells>=cutoff]
            result = p03_test(valid.loc[valid[P03_GROUP]==P03_A,metric],
                              valid.loc[valid[P03_GROUP]==P03_B,metric])
            p03_rows.append(dict(cell_type=ct,min_cells=cutoff,metric=metric,**result))
p03_usage_tests = pd.DataFrame(p03_rows)
p03_usage_tests['FDR'] = p03_usage_tests.groupby(['min_cells','metric'])['pvalue'].transform(p03_bh)
p03_usage_tests.to_csv(p03_out/'usage_disease_tests_sensitivity.csv',index=False)
p03_primary = p03_usage_tests[(p03_usage_tests.min_cells==P03_MIN_CELLS)&
                              (p03_usage_tests.metric=='P03_fraction')].copy()
display(p03_primary.sort_values('delta',ascending=False))

fig, axes = plt.subplots(1,len(P03_TARGETS),figsize=(5*len(P03_TARGETS),4),squeeze=False)
for ax,ct in zip(axes.flat,P03_TARGETS):
    frame = p03_summary[(p03_summary[P03_CT]==ct)&(p03_summary.n_cells>=P03_MIN_CELLS)]
    for x,(group,color) in enumerate([(P03_A,'#377eb8'),(P03_B,'#e76f51')]):
        part=frame[frame[P03_GROUP]==group].sort_values(P03_UNIT)
        jitter=np.linspace(-.12,.12,len(part)) if len(part)>1 else np.zeros(len(part))
        ax.scatter(x+jitter,part.P03_fraction,color=color,s=45)
        for xx,(_,row) in zip(x+jitter,part.iterrows()):
            ax.annotate(str(row[P03_UNIT]),(xx,row.P03_fraction),xytext=(3,3),textcoords='offset points',fontsize=7)
        if len(part): ax.hlines(part.P03_fraction.mean(),x-.2,x+.2,color=color,lw=2)
    ax.set(xticks=[0,1],xticklabels=['Healthy','Severe'],title=ct,ylabel='Mean normalized P03 usage')
fig.tight_layout(); fig.savefig(p03_out/'P03_sample_usage.png',dpi=180,bbox_inches='tight'); plt.show()

fig,ax=plt.subplots(figsize=(11,5))
p03_composition.plot.bar(stacked=True,ax=ax,colormap='tab20')
ax.set_ylabel('Cell fraction'); ax.legend(bbox_to_anchor=(1.02,1),fontsize=8)
fig.tight_layout(); fig.savefig(p03_out/'sample_composition.png',dpi=180,bbox_inches='tight'); plt.show()

p03_forest=p03_primary[p03_primary.status=='tested'].sort_values('delta')
if len(p03_forest):
    fig,ax=plt.subplots(figsize=(7,max(3,len(p03_forest)*.38)))
    y=np.arange(len(p03_forest))
    ax.hlines(y,p03_forest.ci_low,p03_forest.ci_high,color='#377eb8')
    ax.scatter(p03_forest.delta,y,color='#377eb8')
    ax.set(yticks=y,yticklabels=p03_forest.cell_type,xlabel='Severe − Healthy: normalized P03 usage')
    ax.axvline(0,color='gray',ls='--'); fig.tight_layout()
    fig.savefig(p03_out/'P03_disease_effects.png',dpi=180,bbox_inches='tight'); plt.show()

### 2. P03 核心 PAC 与候选基因 wPSI（探索性）
使用全部 P03 核心基因，不使用按方向筛选的 leading genes。保留 wPSI 的 0/1，只按非缺失数过滤；非缺失不等于可靠 read 覆盖。
基因的均值仅使用非缺失细胞，因此同时导出有效细胞数。正式 APA 分析应进一步使用原始 PAC 计数、相同可比较位点和覆盖度控制。
所有目标细胞类型 × 基因的有效检验合并做一次 BH 校正；正 ΔwPSI 仅表示数值增加，需核对上游定义才能解释为远端使用增加。

In [ ]:
p03_identity = gene_tables['P03']
p03_core = sorted(p03_identity.loc[p03_identity.is_core, 'gene_id'])
p03_map = data.get('gene_symbol_map')
def p03_symbol(gid):
    key=str(gid).split('.')[0]
    return str(p03_map.get(key,gid)) if p03_map is not None else str(gid)
p03_pacs = data['feature_anno'].reindex(data['spectra_z'].columns).copy()
p03_pacs['P03_z'] = data['spectra_z'].loc['P03'].reindex(p03_pacs.index)
p03_pacs['P03_weight'] = data['spectra'].loc['P03'].reindex(p03_pacs.index)
p03_pacs['is_P03_core_PAC'] = p03_pacs.P03_z>=CONFIG.get('z_threshold',1.5)
p03_pacs.sort_values('P03_z',ascending=False).to_csv(p03_out/'P03_PAC_annotations.csv')
pd.DataFrame({'gene_id':p03_core,'symbol':[p03_symbol(g) for g in p03_core]}).to_csv(p03_out/'P03_core_genes.csv',index=False)

p03_wpsi_rows=[]
p03_wpsi = data.get('wpsi')
if p03_wpsi is None:
    print('没有加载 wPSI：跳过候选 APA 分析，usage 分析已完成。')
else:
    assert p03_wpsi.index.is_unique and p03_wpsi.columns.is_unique
    p03_genes = pd.Index(p03_core).intersection(p03_wpsi.index)
    print(f'P03核心基因匹配wPSI: {len(p03_genes)}/{len(p03_core)}（精确ID匹配）')
    for (unit,group,ct), frame in p03_cells.groupby(p03_keys,observed=True):
        if ct not in P03_TARGETS or len(frame)<P03_MIN_CELLS: continue
        cell_ids=frame.index.intersection(p03_wpsi.columns)
        block=p03_wpsi.loc[p03_genes,cell_ids].replace([np.inf,-np.inf],np.nan)
        means=block.mean(axis=1); counts=block.notna().sum(axis=1)
        for gid in p03_genes:
            p03_wpsi_rows.append(dict(unit=unit,group=group,cell_type=ct,gene_id=gid,
                symbol=p03_symbol(gid),n_cells=len(frame),n_matrix_cells=len(cell_ids),
                n_valid=int(counts[gid]),valid_fraction=counts[gid]/len(frame),wpsi_mean=means[gid]))
p03_wpsi_samples=pd.DataFrame(p03_wpsi_rows)
p03_gene_tests=pd.DataFrame()
if len(p03_wpsi_samples):
    p03_wpsi_samples.to_csv(p03_out/'P03_gene_sample_wpsi.csv',index=False)
    valid=p03_wpsi_samples[(p03_wpsi_samples.n_valid>=P03_MIN_VALID_WPSI)&
                           (p03_wpsi_samples.valid_fraction>=P03_MIN_VALID_FRAC)]
    rows=[]
    for (ct,gid),frame in valid.groupby(['cell_type','gene_id']):
        result=p03_test(frame.loc[frame.group==P03_A,'wpsi_mean'],frame.loc[frame.group==P03_B,'wpsi_mean'])
        rows.append(dict(cell_type=ct,gene_id=gid,symbol=p03_symbol(gid),**result))
    p03_gene_tests=pd.DataFrame(rows)
    if len(p03_gene_tests):
        p03_gene_tests['FDR']=p03_bh(p03_gene_tests.pvalue)
        p03_gene_tests.to_csv(p03_out/'P03_gene_wpsi_disease_tests.csv',index=False)
        display(p03_gene_tests.sort_values(['FDR','pvalue']).head(30))
        tested=p03_gene_tests[p03_gene_tests.status=='tested'].copy()
        if len(tested):
            top=(tested.assign(abs_delta=tested.delta.abs()).groupby('gene_id').abs_delta.max()
                 .nlargest(20).index)
            heat=tested[tested.gene_id.isin(top)].pivot(index='gene_id',columns='cell_type',values='delta').reindex(top)
            fig,ax=plt.subplots(figsize=(6,max(3,len(heat)*.3)))
            limit=max(.01,float(np.nanmax(np.abs(heat.to_numpy()))))
            im=ax.imshow(np.ma.masked_invalid(heat.to_numpy()),cmap='RdBu_r',vmin=-limit,vmax=limit,aspect='auto')
            ax.set(xticks=np.arange(len(heat.columns)),xticklabels=heat.columns,
                   yticks=np.arange(len(heat)),yticklabels=[p03_symbol(g) for g in heat.index],
                   title='Exploratory wPSI differences (not significance-filtered)')
            fig.colorbar(im,ax=ax,label='Severe − Healthy mean wPSI')
            fig.tight_layout(); fig.savefig(p03_out/'P03_gene_wpsi_effects.png',dpi=180,bbox_inches='tight'); plt.show()
    else: print('没有基因通过有效细胞数过滤。')
else: print('没有可用的样本级 wPSI 记录。')

p03_parameters=dict(program='P03',unit=P03_UNIT,celltype=P03_CT,groups=[P03_A,P03_B],
    min_cells=P03_MIN_CELLS,min_units_per_group=P03_MIN_UNITS,targets=P03_TARGETS,
    min_valid_wpsi=P03_MIN_VALID_WPSI,min_valid_fraction=P03_MIN_VALID_FRAC,seed=P03_SEED,
    usage_primary='row-normalized fraction; sample-equal mean',
    wpsi_method='sample mean of available single-cell wPSI; NOT count-pseudobulk',
    warning='Exploratory; confirm independent donors, batch confounding, PAC coverage and wPSI definition')
(p03_out/'analysis_parameters.json').write_text(json.dumps(p03_parameters,ensure_ascii=False,indent=2))
print('P03分析完成，结果目录:',p03_out)

### 结果阅读顺序
1. `sample_celltype_counts.csv`：确认每组有效样本数量及稀疏细胞类型。
2. `P03_sample_usage.png` / `usage_disease_tests_sensitivity.csv`：看同类细胞内疾病效应、FDR、原始/归一化usage及门槛敏感性。
3. `loo_same_sign`：排查单一样本驱动；不能当作独立验证。
4. `P03_gene_wpsi_disease_tests.csv`：挑选效应稳定、覆盖充分的候选；无显著基因时如实报告。
5. `P03_PAC_annotations.csv`：回到具体位点核对，再用原始PAC计数验证。基因集由同一数据筛选，功能和APA结论都需独立验证。
不要因为未通过FDR就降低阈值追求显著，也不要把正ΔwPSI直接称为UTR延长。